# Chapter 21: Agent Design Patterns — Applying Patterns to GAS

## 🎯 Chapter Goal
Implement 5 canonical agent design patterns and apply each to a GAS (Guardian Angel System) scenario: **ReAct**, **Plan-and-Execute**, **Reflection**, **Tool-Augmented Generation**, and **Multi-step Workflow**.

## 📖 Theory Recap

### The 5 Canonical Agent Design Patterns

| Pattern | Core Idea | Best For |
|---------|-----------|----------|
| **ReAct** | Interleave Reasoning + Acting | Diagnosis, exploration |
| **Plan-and-Execute** | Separate planning from execution | Multi-step tasks |
| **Reflection** | Self-critique and revision | Quality-sensitive outputs |
| **Tool-Augmented** | Structured tool calls | Data retrieval, calculation |
| **Multi-step Workflow** | Sequential pipeline with state | End-to-end processes |

### ReAct (Reasoning + Acting)
Interleaves **Thought** (reasoning) with **Action** (tool use) and **Observation** (tool result):
```
Thought: Glucose is 58 mg/dL — this is hypoglycemia. What's the trend?
Action: check_glucose_trend(patient_id="P001", hours=2)
Observation: {"trend": "falling", "rate": -1.8 mg/dL/min}
Thought: Falling trend + low glucose = urgent intervention needed
Action: lookup_guideline("hypoglycemia treatment")
Observation: ADA 15-15 rule: 15g carbs, wait 15 min, recheck
Answer: URGENT — administer 15g fast-acting carbs immediately
```

### Plan-and-Execute
Separates **what to do** (planning phase, one LLM call) from **how to do it** (execution phase, one call per step). Enables **replanning** when execution fails.

### Reflection
**Generate → Critique → Revise** loop. The agent evaluates its own output against quality criteria and iterates until satisfied or max iterations reached.

### Tool-Augmented Generation
Structured tool calls with typed inputs/outputs. The LLM decides *which* tool to call and *what arguments* to pass — results are injected back into context.

### Multi-step Workflow
Sequential pipeline where each step's output becomes the next step's input. State flows through the pipeline. Simple, predictable, easy to debug.

## 🔨 Build It

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "langgraph", "langchain-openai"],
    check=False
)
print("✅ Dependencies ready")

In [ ]:
import os
import json
import operator
from typing import TypedDict, Annotated
from dataclasses import dataclass, field

# ── Test-mode flag ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL
# ── GAS State ───────────────────────────────────────────────────────────────
class GASState(TypedDict):
    patient_id: str
    glucose_value: float
    alert_level: str
    recommendation: str
    safety_approved: bool
    agent_messages: Annotated[list, operator.add]
    next_agent: str

# ── Mock LLM ────────────────────────────────────────────────────────────────
def mock_llm_call(prompt: str) -> str:
    """Deterministic mock LLM responses for test mode."""
    p = prompt.lower()
    if "critique" in p or "review" in p or "evaluate" in p:
        return json.dumps({
            "is_good_enough": True,
            "feedback": "Recommendation is clinically sound and actionable.",
            "score": 0.92
        })
    elif "plan" in p and "meal" in p:
        return json.dumps([
            "Assess fasting glucose and insulin needs",
            "Plan low-GI breakfast (target: +20-30 mg/dL rise)",
            "Schedule mid-morning snack if glucose < 100 by 10am",
            "Plan balanced lunch with protein and fiber",
            "Evening glucose review and dinner adjustment"
        ])
    elif "execute" in p or "step" in p:
        return "Low-glycemic option recommended. Estimated glucose impact: +25 mg/dL over 2 hours."
    elif "58" in p or "hypoglycemia" in p or "critical" in p:
        return "URGENT: Consume 15g fast-acting carbs immediately. Recheck glucose in 15 minutes."
    elif "285" in p or "245" in p or "hyperglycemia" in p:
        return "Consider correction dose per ISF. Avoid carbohydrates. Increase water intake."
    else:
        return "Glucose within acceptable range. Continue current management plan."

def llm_call(prompt: str) -> str:
    """Call LLM — uses llm_utils backend (Ollama/OpenAI/TEST_MODE)."""
    if TEST_MODE:
        return mock_llm_call(prompt)
    return llm.chat([{"role": "user", "content": prompt}])

print("✅ Setup complete. TEST_MODE =", TEST_MODE)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PATTERN 1: ReAct — GAS Monitor Agent diagnoses a complex glucose pattern
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("PATTERN 1: ReAct (Reasoning + Acting)")
print("=" * 60)

# ── Mock tools available to the ReAct agent ──────────────────────────────────
def lookup_guideline(query: str) -> str:
    db = {
        "hypoglycemia": "ADA 15-15 rule: 15g fast-acting carbs, wait 15 min, recheck. If < 54 mg/dL, call 911.",
        "nocturnal": "ADA: If glucose < 120 mg/dL at bedtime with downward trend, consume 15g snack.",
        "hyperglycemia": "ADA: Glucose > 180 mg/dL post-meal. Consider correction dose per ISF.",
    }
    for key, val in db.items():
        if key in query.lower():
            return val
    return "No specific guideline found. Consult endocrinologist."

def check_glucose_trend(patient_id: str, hours: int = 2) -> dict:
    """Mock: return simulated CGM trend data."""
    return {
        "trend": "falling",
        "rate_mg_dl_per_min": -1.8,
        "readings": [72, 68, 63, 58],
        "hours": hours
    }

def get_patient_context(patient_id: str) -> dict:
    return {
        "age": 45,
        "diabetes_type": "T1D",
        "last_meal": "3 hours ago",
        "insulin_on_board": 0.5
    }

REACT_TOOLS = {
    "lookup_guideline": lookup_guideline,
    "check_glucose_trend": check_glucose_trend,
    "get_patient_context": get_patient_context,
}

# ── ReAct loop ───────────────────────────────────────────────────────────────
def react_monitor_agent(patient_id: str, glucose: float, max_steps: int = 6) -> dict:
    """ReAct agent that diagnoses a glucose situation step by step."""
    trajectory = []

    def log_step(step_type: str, content: str, tool: str = None, result=None):
        entry = {"step": len(trajectory) + 1, "type": step_type, "content": content}
        if tool:
            entry["tool"] = tool
            entry["result"] = result
        trajectory.append(entry)
        icon = {"Thought": "💭", "Action": "🔧", "Observation": "👁️ ", "Answer": "✅"}.get(step_type, "  ")
        print(f"  {icon} {step_type}: {content[:90]}")
        if result:
            print(f"       → {str(result)[:80]}")

    # Step 1: Initial reasoning
    log_step("Thought", f"Glucose is {glucose} mg/dL. Need to assess severity and context.")

    # Step 2: Get patient context
    ctx = REACT_TOOLS["get_patient_context"](patient_id)
    log_step("Action", f"get_patient_context(patient_id='{patient_id}')", "get_patient_context", ctx)

    # Step 3: Reason about context
    log_step("Thought", f"Patient is {ctx['age']}y T{ctx['diabetes_type'][-1]}D, IOB={ctx['insulin_on_board']}U. Check trend.")

    # Step 4: Check trend
    trend = REACT_TOOLS["check_glucose_trend"](patient_id, hours=2)
    log_step("Action", f"check_glucose_trend(patient_id='{patient_id}', hours=2)", "check_glucose_trend", trend)

    # Step 5: Reason about trend
    log_step("Thought",
        f"Trend is {trend['trend']} at {trend['rate_mg_dl_per_min']} mg/dL/min. "
        f"Readings: {trend['readings']}. Concerning — need guideline."
    )

    # Step 6: Look up guideline
    query = "hypoglycemia" if glucose < 70 else "hyperglycemia"
    guideline = REACT_TOOLS["lookup_guideline"](query)
    log_step("Action", f"lookup_guideline(query='{query}')", "lookup_guideline", guideline)

    # Final answer
    severity = "Severe hypoglycemia" if glucose < 54 else ("Hypoglycemia" if glucose < 70 else "Hyperglycemia")
    answer = f"{severity} with {trend['trend']} trend ({trend['rate_mg_dl_per_min']} mg/dL/min). {guideline}"
    log_step("Answer", answer)

    return {"answer": answer, "trajectory": trajectory, "steps": len(trajectory)}

# Run ReAct on hypoglycemia scenario
result = react_monitor_agent("P001", glucose=58.0)
print(f"\n📊 ReAct completed in {result['steps']} steps")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PATTERN 2: Plan-and-Execute — GAS Advisor plans a full day meal schedule
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("PATTERN 2: Plan-and-Execute")
print("=" * 60)

def plan_day_meals(patient_id: str, fasting_glucose: float) -> list:
    """PLANNING PHASE: generate a structured meal plan."""
    prompt = (
        f"Patient {patient_id} has fasting glucose {fasting_glucose} mg/dL. "
        "Create a JSON list of 5 meal planning steps for the day."
    )
    response = llm_call(prompt)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        # Fallback: split by newlines
        return [line.strip() for line in response.split("\n") if line.strip()][:5]

def execute_meal_step(step: str, patient_id: str, current_glucose: float, step_num: int) -> dict:
    """EXECUTION PHASE: execute a single meal planning step."""
    prompt = (
        f"Execute this meal planning step for patient {patient_id} "
        f"(current glucose={current_glucose} mg/dL): {step}"
    )
    result = llm_call(prompt)
    # Simulate glucose impact
    glucose_impacts = [0, +25, +10, +35, +20]
    impact = glucose_impacts[step_num % len(glucose_impacts)]
    return {
        "step": step,
        "result": result,
        "glucose_impact_mg_dl": impact,
        "new_glucose": current_glucose + impact
    }

# ── PLANNING PHASE ───────────────────────────────────────────────────────────
print("\n📋 PLANNING PHASE:")
plan = plan_day_meals("P001", fasting_glucose=145.0)
for i, step in enumerate(plan, 1):
    print(f"  Step {i}: {step}")

# ── EXECUTION PHASE ──────────────────────────────────────────────────────────
print("\n⚙️  EXECUTION PHASE:")
execution_results = []
current_glucose = 145.0

for i, step in enumerate(plan, 1):
    print(f"\n  Executing Step {i}: {step[:55]}...")
    result = execute_meal_step(step, "P001", current_glucose, i)
    execution_results.append(result)
    current_glucose = result["new_glucose"]
    print(f"  Result: {result['result'][:70]}")
    print(f"  Glucose: {result['new_glucose'] - result['glucose_impact_mg_dl']:.0f} → {result['new_glucose']:.0f} mg/dL")

print(f"\n✅ Plan-and-Execute: {len(plan)} steps planned, {len(execution_results)} executed")
print(f"   Final glucose: {current_glucose:.0f} mg/dL")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PATTERN 3: Reflection — GAS Advisor generates, critiques, and revises
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("PATTERN 3: Reflection")
print("=" * 60)

def generate_recommendation(state: dict) -> str:
    """Generate initial recommendation."""
    feedback_clause = ""
    if state.get("previous_feedback"):
        feedback_clause = f" Previous feedback: {state['previous_feedback']}. Improve accordingly."
    prompt = (
        f"Patient glucose: {state['glucose_value']} mg/dL, alert: {state['alert_level']}. "
        f"Provide a clinical recommendation.{feedback_clause}"
    )
    return llm_call(prompt)

def critique_recommendation(recommendation: str, state: dict) -> dict:
    """Critique the recommendation for quality and safety."""
    prompt = (
        f"Critique this diabetes recommendation for patient with glucose={state['glucose_value']} mg/dL, "
        f"alert={state['alert_level']}.\n"
        f"Recommendation: {recommendation}\n"
        'Return JSON: {"is_good_enough": bool, "feedback": str, "score": float}'
    )
    response = llm_call(prompt)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return {"is_good_enough": True, "feedback": "Clinically appropriate.", "score": 0.85}

def reflection_loop(state: dict, max_iterations: int = 3) -> dict:
    """Reflection pattern: generate → critique → revise until good enough."""
    history = []
    recommendation = ""
    critique = {}

    for i in range(max_iterations):
        print(f"\n  🔄 Iteration {i+1}/{max_iterations}")

        # Generate
        recommendation = generate_recommendation(state)
        print(f"  📝 Generated: {recommendation[:80]}...")

        # Critique
        critique = critique_recommendation(recommendation, state)
        print(f"  🔍 Score: {critique['score']:.2f} | Good enough: {critique['is_good_enough']}")
        print(f"  💬 Feedback: {critique['feedback'][:70]}")

        history.append({
            "iteration": i + 1,
            "recommendation": recommendation,
            "critique": critique
        })

        if critique["is_good_enough"]:
            print(f"  ✅ Accepted after {i+1} iteration(s)")
            break

        # Update state with feedback for next iteration
        state = {**state, "previous_feedback": critique["feedback"]}

    return {
        "final_recommendation": recommendation,
        "iterations": len(history),
        "final_score": critique.get("score", 0.0),
        "history": history
    }

# Run reflection on a critical hypoglycemia scenario
state = {
    "patient_id": "P002",
    "glucose_value": 58.0,
    "alert_level": "critical",
}

result = reflection_loop(state, max_iterations=3)
print(f"\n📊 Reflection Summary:")
print(f"   Iterations   : {result['iterations']}")
print(f"   Final score  : {result['final_score']:.2f}")
print(f"   Final rec    : {result['final_recommendation'][:100]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PATTERN 4: Tool-Augmented Generation — GAS Advisor with structured tools
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("PATTERN 4: Tool-Augmented Generation")
print("=" * 60)

# ── Structured tool definitions ──────────────────────────────────────────────
def lookup_guideline_tool(query: str) -> str:
    """Search ADA clinical guidelines."""
    db = {
        "correction dose": "Correction dose = (Current BG - Target BG) / ISF. Typical ISF: 30-50 mg/dL per unit.",
        "hypoglycemia treatment": "15-15 rule: 15g fast-acting carbs, wait 15 min, recheck. Repeat if still < 70.",
        "target glucose": "ADA target: 80-130 mg/dL fasting, < 180 mg/dL post-meal (1-2h after eating).",
        "exercise": "Check glucose before exercise. If < 90 mg/dL, consume 15-30g carbs before starting.",
    }
    for key, val in db.items():
        if key in query.lower():
            return val
    return f"Guideline for '{query}': Consult ADA Standards of Medical Care in Diabetes."

def calculate_dose_tool(current_glucose: float, target_glucose: float, isf: float) -> dict:
    """Calculate insulin correction dose."""
    dose = (current_glucose - target_glucose) / isf
    dose = max(0.0, round(dose, 1))  # No negative doses
    return {
        "correction_dose_units": dose,
        "current_glucose": current_glucose,
        "target_glucose": target_glucose,
        "isf": isf,
        "formula": f"({current_glucose} - {target_glucose}) / {isf} = {dose}U"
    }

def check_interactions_tool(medication: str, glucose: float) -> dict:
    """Check drug-glucose interactions."""
    interactions = {
        "metformin": {"effect": "neutral", "note": "No hypoglycemia risk when used alone."},
        "insulin": {"effect": "lowers", "note": "Monitor closely for hypoglycemia."},
        "steroids": {"effect": "raises", "note": "May cause steroid-induced hyperglycemia."},
        "glipizide": {"effect": "lowers", "note": "Sulfonylurea — hypoglycemia risk, especially if meal skipped."},
    }
    for med, info in interactions.items():
        if med in medication.lower():
            return {**info, "medication": medication, "current_glucose": glucose}
    return {"effect": "unknown", "note": "No interaction data available.", "medication": medication}

TOOLS = {
    "lookup_guideline": lookup_guideline_tool,
    "calculate_dose": calculate_dose_tool,
    "check_interactions": check_interactions_tool,
}

def tool_augmented_advisor(patient_id: str, glucose: float, medications: list) -> dict:
    """Advisor agent that uses structured tool calls to build a recommendation."""
    tool_calls = []

    # Tool call 1: Look up relevant guideline
    print("  🔧 Tool call 1: lookup_guideline")
    query = "correction dose" if glucose > 180 else "hypoglycemia treatment"
    r1 = TOOLS["lookup_guideline"](query=query)
    tool_calls.append({"tool": "lookup_guideline", "args": {"query": query}, "result": r1})
    print(f"     → {r1[:80]}")

    # Tool call 2: Calculate dose (if hyperglycemia)
    dose_result = None
    if glucose > 180:
        print("  🔧 Tool call 2: calculate_dose")
        dose_result = TOOLS["calculate_dose"](current_glucose=glucose, target_glucose=120.0, isf=50.0)
        tool_calls.append({"tool": "calculate_dose", "args": {"current_glucose": glucose}, "result": dose_result})
        print(f"     → {dose_result['formula']}")

    # Tool call 3: Check interactions for each medication
    for med in medications:
        print(f"  🔧 Tool call 3: check_interactions({med})")
        r3 = TOOLS["check_interactions"](medication=med, glucose=glucose)
        tool_calls.append({"tool": "check_interactions", "args": {"medication": med}, "result": r3})
        print(f"     → {r3['effect'].upper()}: {r3['note']}")

    # Synthesize recommendation from tool results
    parts = []
    if dose_result and dose_result["correction_dose_units"] > 0:
        parts.append(f"Administer {dose_result['correction_dose_units']}U correction insulin.")
    parts.append(r1)
    recommendation = " ".join(parts)

    return {
        "recommendation": recommendation,
        "tool_calls": tool_calls,
        "tools_used": [tc["tool"] for tc in tool_calls]
    }

# Run tool-augmented advisor on hyperglycemia
result = tool_augmented_advisor("P003", glucose=245.0, medications=["insulin", "metformin"])
print(f"\n📊 Tool-Augmented Summary:")
print(f"   Tools used    : {result['tools_used']}")
print(f"   Recommendation: {result['recommendation'][:100]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PATTERN 5: Multi-step Workflow — Full GAS pipeline as sequential steps
# ═══════════════════════════════════════════════════════════════════════════

print("=" * 60)
print("PATTERN 5: Multi-step Workflow")
print("=" * 60)

@dataclass
class WorkflowState:
    patient_id: str
    glucose_value: float
    alert_level: str = "none"
    recommendation: str = ""
    safety_approved: bool = False
    caregiver_notified: bool = False
    step_log: list = field(default_factory=list)

    def log(self, step: str, msg: str):
        entry = f"[{step}] {msg}"
        self.step_log.append(entry)
        print(f"  {entry}")

# ── Pipeline step functions ──────────────────────────────────────────────────
def step_monitor(state: WorkflowState) -> WorkflowState:
    g = state.glucose_value
    if g < 54:     state.alert_level = "critical"
    elif g < 70:   state.alert_level = "high"
    elif g < 80:   state.alert_level = "medium"
    elif g > 250:  state.alert_level = "critical"
    elif g > 180:  state.alert_level = "high"
    elif g > 140:  state.alert_level = "medium"
    else:          state.alert_level = "none"
    state.log("Monitor", f"glucose={g} mg/dL → alert={state.alert_level}")
    return state

def step_advise(state: WorkflowState) -> WorkflowState:
    if state.alert_level == "none":
        state.recommendation = "Continue current management. No action needed."
    elif state.alert_level in ("critical", "high") and state.glucose_value < 70:
        state.recommendation = "URGENT: 15g fast-acting carbs NOW. Recheck in 15 min."
    elif state.alert_level in ("critical", "high") and state.glucose_value > 180:
        state.recommendation = "Correction dose indicated. Avoid carbs. Increase hydration."
    else:
        state.recommendation = "Monitor closely. Recheck in 30 minutes."
    state.log("Advisor", f"recommendation set ({len(state.recommendation)} chars)")
    return state

def step_safety_check(state: WorkflowState) -> WorkflowState:
    rec = state.recommendation.lower()
    is_safe = "ignore" not in rec and "no treatment" not in rec
    if state.alert_level == "critical":
        is_safe = is_safe and any(kw in rec for kw in ["urgent", "correction", "immediately", "now"])
    state.safety_approved = is_safe
    state.log("Safety", f"approved={is_safe}")
    return state

def step_notify(state: WorkflowState) -> WorkflowState:
    if state.alert_level in ("critical", "high") and state.safety_approved:
        state.caregiver_notified = True
        state.log("Notifier", f"🚨 Caregiver alerted for patient {state.patient_id}")
    else:
        state.log("Notifier", "No notification needed")
    return state

def run_gas_workflow(patient_id: str, glucose: float) -> WorkflowState:
    """Run the full GAS pipeline as a sequential multi-step workflow."""
    state = WorkflowState(patient_id=patient_id, glucose_value=glucose)
    pipeline = [step_monitor, step_advise, step_safety_check, step_notify]
    for step_fn in pipeline:
        state = step_fn(state)
    return state

# Run on all 3 scenarios
scenarios = [
    ("P001", 105.0, "Normal"),
    ("P002", 58.0,  "Hypoglycemia"),
    ("P003", 285.0, "Hyperglycemia"),
]

workflow_results = []
for patient_id, glucose, name in scenarios:
    print(f"\n{'─'*50}")
    print(f"🧪 {name} (glucose={glucose} mg/dL)")
    state = run_gas_workflow(patient_id, glucose)
    workflow_results.append(state)
    print(f"  📊 alert={state.alert_level}, approved={state.safety_approved}, notified={state.caregiver_notified}")

print("\n✅ Multi-step Workflow pattern complete")

In [ ]:
# ── Pattern Decision Guide ───────────────────────────────────────────────────

print("=" * 60)
print("PATTERN DECISION GUIDE")
print("=" * 60)

decision_guide = [
    {
        "pattern": "ReAct",
        "use_when": "Diagnosis, exploration, unknown problem structure",
        "gas_example": "Monitor Agent diagnosing complex glucose patterns",
        "pros": "Flexible, handles unexpected situations",
        "cons": "Slower, harder to test deterministically"
    },
    {
        "pattern": "Plan-and-Execute",
        "use_when": "Multi-step tasks with known structure",
        "gas_example": "Advisor planning full-day meal schedule",
        "pros": "Efficient, allows replanning on failure",
        "cons": "Plan may be wrong; requires good planner"
    },
    {
        "pattern": "Reflection",
        "use_when": "Quality-sensitive, safety-critical outputs",
        "gas_example": "Advisor refining insulin dosing recommendations",
        "pros": "Higher quality output, self-correcting",
        "cons": "Multiple LLM calls, higher latency"
    },
    {
        "pattern": "Tool-Augmented",
        "use_when": "Structured data retrieval, calculations, API calls",
        "gas_example": "Advisor calling calculate_dose, lookup_guideline",
        "pros": "Precise, auditable, composable",
        "cons": "Requires well-defined tool schemas"
    },
    {
        "pattern": "Multi-step Workflow",
        "use_when": "End-to-end processes with clear sequential steps",
        "gas_example": "Full GAS pipeline: monitor → advise → safety → notify",
        "pros": "Simple, predictable, easy to debug",
        "cons": "Rigid; hard to handle branching logic"
    },
]

for info in decision_guide:
    print(f"\n  📌 {info['pattern']}")
    for key, val in info.items():
        if key != "pattern":
            print(f"     {key:<12}: {val}")

## 🏥 GAS Connection

The GAS system uses **different patterns for different scenarios** — no single pattern fits all clinical situations:

| GAS Agent | Pattern Used | Reason |
|-----------|-------------|--------|
| Monitor Agent | **ReAct** | Diagnosis requires iterative reasoning + tool use |
| Advisor Agent (complex) | **Reflection** | Safety-critical recommendations need self-critique |
| Advisor Agent (meal plan) | **Plan-and-Execute** | Day planning has known multi-step structure |
| Advisor Agent (dosing) | **Tool-Augmented** | Dose calculation requires precise, auditable tool calls |
| Full GAS Pipeline | **Multi-step Workflow** | End-to-end coordination is sequential |

**Key insight:** The GAS system is a *portfolio* of patterns, each applied where it fits best. LangGraph's conditional routing (Chapter 20) lets us switch patterns dynamically based on the clinical situation — e.g., use Reflection only for critical alerts, use Multi-step Workflow for routine monitoring.

## ✅ Checkpoint

**What we built:**
- ✅ **ReAct**: Monitor Agent diagnoses hypoglycemia with 6-step Thought/Action/Observation trajectory
- ✅ **Plan-and-Execute**: Advisor plans a full-day meal schedule in 2 phases (plan once, execute per step)
- ✅ **Reflection**: Advisor iterates recommendation until critique score meets threshold
- ✅ **Tool-Augmented**: Advisor calls 3 structured tools (guideline lookup, dose calculation, drug interactions)
- ✅ **Multi-step Workflow**: Full GAS pipeline runs all 3 clinical scenarios sequentially
- ✅ **Decision Guide**: When to use each pattern in GAS context

**Next:** Chapter 22 — Agentic Environments & Benchmarks (building a simulation environment to evaluate GAS performance)